# Code à répétition 3 qubits

$$|0\rangle \to |0_L\rangle = |000\rangle, \quad |1\rangle \to |1_L\rangle = |111\rangle$$

Correction d'erreurs de flip $X$ (bit-flip) avec détection de syndrome.

In [ ]:
import numpy as np
import qutip as qt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, Aer, execute
from qiskit.visualization import plot_histogram
import stim
import matplotlib.pyplot as plt

### Circuit encodeur (Qiskit)

Prépare $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ dans l'état logique:
$$\alpha|000\rangle + \beta|111\rangle$$

In [ ]:
def encode_3qubit(qc, q, a):
    qc.cx(q[0], q[1])
    qc.cx(q[0], q[2])
    return qc

def syndrome_3qubit(qc, q, c):
    qc.cx(q[0], q[1])
    qc.cx(q[0], q[2])
    qc.cx(q[1], q[0])
    qc.cx(q[1], q[2])
    qc.measure(q[0], c[0])
    qc.measure(q[2], c[1])
    return qc

def correct_3qubit(qc, q, c):
    qc.x(q[1]).c_if(c, 3)
    qc.x(q[0]).c_if(c, 1)
    qc.x(q[2]).c_if(c, 2)
    return qc

In [ ]:
# Test du code complet
q = QuantumRegister(3, 'q')
c = ClassicalRegister(2, 'syndrome')
c_out = ClassicalRegister(1, 'out')
qc = QuantumCircuit(q, c, c_out)

qc.h(0)
encode_3qubit(qc, q, c)
qc.x(0)
syndrome_3qubit(qc, q, c)
correct_3qubit(qc, q, c)
qc.measure(q[0], c_out[0])

backend = Aer.get_backend('qasm_simulator')
counts = execute(qc, backend, shots=1024).result().get_counts()
plot_histogram(counts, title='Correction: bit-flip sur q0 → résultat attendu 0')

### Modèle QuTiP: syndrome et correction

États du code à répétition avec QuTiP:

$$|0_L\rangle = |000\rangle, \quad |1_L\rangle = |111\rangle$$

Les projecteurs de syndrome: $P_0 = |000\rangle\langle000| + |111\rangle\langle111|$

In [ ]:
zero = qt.tensor(qt.basis(2, 0), qt.basis(2, 0), qt.basis(2, 0))
one = qt.tensor(qt.basis(2, 1), qt.basis(2, 1), qt.basis(2, 1))
psi_log = (zero + one).unit()

print('|0_L⟩:', zero)
print('|1_L⟩:', one)
print('|+_L⟩:', psi_log)

In [ ]:
# Opérateurs d'erreur X sur chaque qubit
X1 = qt.tensor(qt.sigmax(), qt.qeye(2), qt.qeye(2))
X2 = qt.tensor(qt.qeye(2), qt.sigmax(), qt.qeye(2))
X3 = qt.tensor(qt.qeye(2), qt.qeye(2), qt.sigmax())

# Syndrome: Z1Z2 et Z2Z3
Z1Z2 = qt.tensor(qt.sigmaz(), qt.sigmaz(), qt.qeye(2))
Z2Z3 = qt.tensor(qt.qeye(2), qt.sigmaz(), qt.sigmaz())

psi_err = X1 * zero
print('État |000⟩ après X₁:', psi_err)
print('<Z₁Z₂>:', (psi_err.dag() * Z1Z2 * psi_err).real)
print('<Z₂Z₃>:', (psi_err.dag() * Z2Z3 * psi_err).real)

In [ ]:
# Correction: appliquer X_i en fonction du syndrome
def correct_qubit(state, syndrome_val):
    if syndrome_val == 0:
        return state
    elif syndrome_val == 1:
        return X3 * state
    elif syndrome_val == 2:
        return X1 * state
    else:
        return X2 * state

for err_op, name in [(X1, 'X1'), (X2, 'X2'), (X3, 'X3')]:
    err_state = err_op * zero
    s1 = (err_state.dag() * Z1Z2 * err_state).real
    s2 = (err_state.dag() * Z2Z3 * err_state).real
    syn = int((1 - s1) / 2) * 2 + int((1 - s2) / 2)
    corrected = correct_qubit(err_state, syn)
    fid = (zero.dag() * corrected).real**2
    print(f'Erreur {name}: syndrome={syn}, fidélité={fid:.3f}')

### Modèle Stim: stabilisateur

Code à répétition comme code stabilisateur:
- $S_1 = Z_1 Z_2$
- $S_2 = Z_2 Z_3$
- Générateurs logiques: $\bar{X} = X_1 X_2 X_3$, $\bar{Z} = Z_1$

In [ ]:
circ = stim.Circuit()
circ.append('H', 0)
circ.append('CNOT', [0, 1])
circ.append('CNOT', [0, 2])
circ.append('X_ERROR', 0, 0.5)
circ.append('MR', [0, 1, 2])
sampler = circ.compile_sampler()
samples = sampler.sample(10)
print('Échantillons (H, CNOT, X_ERROR, mesure):')
print(samples)

In [ ]:
# Détecteur dans Stim
circuit = stim.Circuit()
circuit.append('H', 0)
circuit.append('CNOT', [0, 1])
circuit.append('CNOT', [0, 2])
circuit.append('X_ERROR', 0, 0.3)
circuit.append('CNOT', [0, 1])
circuit.append('CNOT', [0, 2])
circuit.append('M', [0, 1, 2])

detector = circuit.detector_error_model()
print('Modèle d\'erreur du détecteur:')
print(detector)

## Questions

**Q1.** Ajouter au circuit Qiskit la correction d'une erreur $X$ sur chacun des 3 qubits (q0, q1, q2). Vérifier que la correction restaure l'état initial dans les 3 cas. Mesurer la fidélité.

**Q2.** Avec QuTiP, simuler un canal de bit-flip $\mathcal{E}(\rho) = (1-p)\rho + p X\rho X$ appliqué de manière indépendante sur chaque qubit. Tracer la fidélité de l'état corrigé en fonction de $p \in [0, 0.5]$.

In [ ]:
# Q2: Fidélité vs probabilité d'erreur
p_vals = np.linspace(0, 0.5, 20)
fids = []

for p in p_vals:
    psi_in = zero
    rho = qt.ket2dm(psi_in)
    for k in range(3):
        Xk = [qt.tensor(*[qt.sigmax() if i == k else qt.qeye(2) for i in range(3)])][0]
        rho = (1 - p) * rho + p * Xk * rho * Xk
    s1 = (psi_in.dag() * rho * psi_in).real
    fids.append(s1)

plt.plot(p_vals, fids, linewidth=2)
plt.xlabel('Probabilité d\'erreur p')
plt.ylabel('Fidélité')
plt.title('Fidélité après correction vs taux d\'erreur')
plt.grid(True)
plt.show()